# 3.06 Weaviate Vector Store

**Weaviate**는 Go 기반 벡터 DB. 관리형(Weaviate Cloud)과 셀프호스팅(Docker) 모두 지원하며, **하이브리드 검색**(BM25 + dense)을 `alpha` 가중치로 쉽게 제어할 수 있다.

## 학습 목표

- `weaviate.connect_to_weaviate_cloud(...)` 또는 `connect_to_local()` 로 클라이언트 초기화
- `WeaviateVectorStore` 로 컬렉션(스키마) 자동 생성
- Weaviate GraphQL `where` DSL로 메타데이터 필터
- `similarity_search(alpha=0.5)` 하이브리드 검색

## 언제 쓰나

- 하이브리드 검색(semantic + keyword)을 벡터 스토어 레벨에서 필요할 때
- 모듈 시스템(text2vec, generative 모듈)으로 검색·생성을 서버 쪽에서 처리
- 다중 테넌트 (Weaviate의 tenants 기능)

## 3.06.1 환경 설정 / 서비스 연결

두 가지 경로:

1. Weaviate Cloud: `.env`에 `WEAVIATE_URL` + `WEAVIATE_API_KEY`
2. 로컬 Docker: `docker run -p 8080:8080 -p 50051:50051 cr.weaviate.io/semitechnologies/weaviate:latest`

probe 셀은 `WEAVIATE_URL` 이 있으면 Cloud로, 없으면 local로 접속을 시도한다.

In [ ]:
# !pip install -U langchain-weaviate weaviate-client langchain-openai

import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요"

import weaviate
from weaviate.classes.init import Auth

WEAVIATE_URL = os.environ.get("WEAVIATE_URL")
WEAVIATE_API_KEY = os.environ.get("WEAVIATE_API_KEY")

if WEAVIATE_URL and WEAVIATE_API_KEY:
    # Weaviate Cloud
    client = weaviate.connect_to_weaviate_cloud(
        cluster_url=WEAVIATE_URL,
        auth_credentials=Auth.api_key(WEAVIATE_API_KEY),
        headers={"X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"]},
    )
else:
    # 로컬 Docker
    client = weaviate.connect_to_local()

assert client.is_ready(), "Weaviate 서버에 연결할 수 없다"
print("Weaviate ready:", client.get_meta()["version"])

## 3.06.2 기본 사용법

`WeaviateVectorStore.from_documents(docs, embedding, client, index_name, text_key="text")` 로 컬렉션을 만든다. `index_name` 은 Weaviate의 **Collection** 이름과 동일 (첫 글자는 대문자 권장).

In [ ]:
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_weaviate import WeaviateVectorStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

docs = [
    Document(page_content="Weaviate는 하이브리드 검색을 기본 제공하는 벡터 DB이다.", metadata={"source": "weaviate", "tier": "hybrid"}),
    Document(page_content="BM25는 전통적인 keyword 기반 스코어링 함수이다.", metadata={"source": "paper", "tier": "keyword"}),
    Document(page_content="RRF(Reciprocal Rank Fusion)로 dense와 sparse 결과를 결합한다.", metadata={"source": "paper", "tier": "fusion"}),
    Document(page_content="HNSW는 Weaviate의 기본 ANN 인덱스이다.", metadata={"source": "docs", "tier": "dense"}),
]

INDEX = "VSDemo"

# 이전 실행 흔적 제거
if client.collections.exists(INDEX):
    client.collections.delete(INDEX)

store = WeaviateVectorStore.from_documents(
    docs,
    embedding=embeddings,
    client=client,
    index_name=INDEX,
    text_key="text",
)
for d in store.similarity_search("하이브리드 검색", k=3):
    print("-", d.metadata.get("source"), "|", d.page_content[:50])

## 3.06.3 메타데이터 필터링 — Weaviate `Filter` DSL

Weaviate v4 클라이언트는 타입 안정적인 `Filter` 객체를 쓴다. LangChain store에는 `filters=` 인자로 넘긴다.

- `Filter.by_property("source").equal("weaviate")`
- `Filter.by_property("tier").contains_any(["dense", "hybrid"])`
- `Filter.all_of([...])` / `Filter.any_of([...])`

In [ ]:
from weaviate.classes.query import Filter

flt = Filter.by_property("tier").contains_any(["dense", "hybrid"])

hits = store.similarity_search("벡터 인덱스", k=3, filters=flt)
for d in hits:
    print("-", d.metadata, "|", d.page_content[:50])

## 3.06.4 점수 포함 · MMR

In [ ]:
print("--- with_score ---")
for doc, score in store.similarity_search_with_score("BM25", k=3):
    print(f"{score:.4f}  {doc.metadata.get('source')}")

print("\n--- MMR ---")
for d in store.max_marginal_relevance_search("검색 엔진", k=3, fetch_k=4, lambda_mult=0.4):
    print("-", d.metadata.get("source"))

## 3.06.5 하이브리드 검색

Weaviate의 `similarity_search(..., alpha=...)` 는 **하이브리드 스코어**를 계산한다.

- `alpha=0.0` — 순수 BM25 (keyword)
- `alpha=1.0` — 순수 dense (vector)
- `alpha=0.5` — 절반씩 섞기 (RRF)

같은 질의에서 alpha 값에 따라 결과가 어떻게 바뀌는지 비교한다.

In [ ]:
query = "HNSW 인덱스"

for a in [0.0, 0.5, 1.0]:
    print(f"\nalpha={a}")
    for d in store.similarity_search(query, k=3, alpha=a):
        print("-", d.metadata.get("source"), "|", d.page_content[:50])

## 3.06.6 정리

데모 컬렉션 삭제 + 클라이언트 close.

In [ ]:
client.collections.delete(INDEX)
client.close()
print("cleaned:", INDEX)

## 체크리스트

- [ ] Collection(인덱스) 이름은 **첫 글자 대문자** (Weaviate 규칙)
- [ ] Cloud 호출 시 `X-OpenAI-Api-Key` 헤더 필요 (text2vec-openai 모듈 쓸 때)
- [ ] `alpha` 로 dense/BM25 가중치 조정 — 쿼리별로 적응적으로 바꿀 수 있다
- [ ] 클라이언트는 **반드시 `client.close()`** — 미종료 시 gRPC connection 누수

## 다음

- `07_milvus.ipynb` — 대규모 분산 환경
- `05_retrievers/` — retriever + rerank 조합

## 참고

- Weaviate docs: https://weaviate.io/developers/weaviate
- `langchain-weaviate`: https://python.langchain.com/docs/integrations/vectorstores/weaviate